# 07 — Decoupled-dim Linear-Attention DT + Inference Movie

Linear-attention companion to `06_train_dt_decouple.ipynb`. Same dataset, same hyperparams (`state_dim=60, embed_dim=128, num_heads=2 → d_head=64`); the only meaningful difference is the attention mechanism.

Vs. `05_train_linearattn_dt.ipynb`, two changes:
  * `state_dim` decoupled from `embed_dim` (lets us widen past the 60-D encoder cap).
  * Feature map swapped from `elu(x) + 1` to `relu(x) + eps`.

## 1. Setup

In [1]:
from pathlib import Path
import json, time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import sys; sys.path.insert(0, '../scripts')
import train_dt as T

from corner_maze_rl.encoders.grid_cells import GridCellEncoder
from corner_maze_rl.env.corner_maze_env import CornerMazeEnv
from corner_maze_rl.models.linear_decision_transformer_decoupled_dimension import (
    LinearDTConfigDecoupled, LinearDecisionTransformerDecoupled,
)

DATA_PATH    = Path('../data/yoked/dataset/actions_synthetic_pretrial.parquet')
RUN_DIR      = Path('../runs/dt/nb07'); RUN_DIR.mkdir(parents=True, exist_ok=True)

CONTEXT_SIZE = 32
STATE_DIM    = 60
EMBED_DIM    = 128
NUM_HEADS    = 2
NUM_LAYERS   = 2
POS_ENCODING = 'learned'
LR           = 1e-3
WEIGHT_DECAY = 1e-4
BATCH_SIZE   = 1024     # bumped from 64 — the model+context is tiny so 64 left
                        # the 4090 launch-bound at 50W. 1024 gives ~16x more work
                        # per kernel launch with negligible VRAM cost.
EPOCHS       = 10
VAL_FRAC     = 0.10
SEED         = 0
MAX_SESSIONS = None

device = ('cuda' if torch.cuda.is_available()
          else 'mps' if torch.backends.mps.is_available()
          else 'cpu')
torch.manual_seed(SEED); np.random.seed(SEED)
print(f'device={device}  d_head={EMBED_DIM // NUM_HEADS}')

device=cuda  d_head=64


## 2. Load + window the yoked dataset

In [2]:
df = pd.read_parquet(DATA_PATH)
if MAX_SESSIONS is not None:
    keep = sorted(df['session_id'].unique())[:MAX_SESSIONS]
    df = df[df['session_id'].isin(keep)].reset_index(drop=True)
print(f'rows={len(df):,}  sessions={df["session_id"].nunique()}')

encoder = GridCellEncoder()
assert encoder.output_dim == STATE_DIM

# The maze has only 11*11*4 = 484 distinct (x, y, direction) poses.
# Store one int16 pose ID per step and expand to a STATE_DIM-vector on-GPU at
# batch time — cuts dataset RAM from ~30 GB to ~250 MB and the build from
# ~minute to seconds. The model's input projection lifts STATE_DIM -> EMBED_DIM.
N_X, N_Y, N_D = 11, 11, 4
N_POSES = N_X * N_Y * N_D
pose_table_np = np.zeros((N_POSES, STATE_DIM), dtype=np.float32)
for x in range(1, N_X + 1):
    for y in range(1, N_Y + 1):
        for d in range(N_D):
            pose_table_np[((x - 1) * N_Y + (y - 1)) * N_D + d] = encoder.encode(x, y, d)
pose_table = torch.from_numpy(pose_table_np).to(device)

def encode_session_ids(sdf):
    xs = sdf['grid_x'].to_numpy(np.int32)
    ys = sdf['grid_y'].to_numpy(np.int32)
    ds = sdf['direction'].to_numpy(np.int32)
    pose_ids = (((xs - 1) * N_Y + (ys - 1)) * N_D + ds).astype(np.int16)
    rewarded = sdf['rewarded'].to_numpy(np.float32)
    rtg = np.flip(np.cumsum(np.flip(rewarded))).astype(np.float32)
    action_ids = sdf['action'].to_numpy(np.int64).clip(0, 4).astype(np.int8)
    return pose_ids, rtg, action_ids

def window_1d(arr, C):
    padded = np.concatenate([np.zeros(C - 1, dtype=arr.dtype), arr])
    return np.lib.stride_tricks.sliding_window_view(padded, C).copy()  # (n, C)

acc_pose, acc_rtg, acc_act = [], [], []
for sid in df['session_id'].unique():
    sdf = df[df['session_id'] == sid].sort_values('step')
    if len(sdf) == 0:
        continue
    pose_ids, rtg, action_ids = encode_session_ids(sdf)
    acc_pose.append(window_1d(pose_ids, CONTEXT_SIZE))
    acc_rtg.append(window_1d(rtg, CONTEXT_SIZE))
    acc_act.append(window_1d(action_ids, CONTEXT_SIZE))

pose_w = torch.from_numpy(np.concatenate(acc_pose, axis=0))   # (N, C) int16
rtg_w  = torch.from_numpy(np.concatenate(acc_rtg, axis=0))    # (N, C) float32
act_w  = torch.from_numpy(np.concatenate(acc_act, axis=0))    # (N, C) int8
del acc_pose, acc_rtg, acc_act

full = torch.utils.data.TensorDataset(rtg_w, pose_w, act_w)
n = len(full); n_val = max(1, int(n * VAL_FRAC)); n_train = n - n_val
train_ds, val_ds = torch.utils.data.random_split(
    full, [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED),
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          drop_last=True, pin_memory=(device == 'cuda'))
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          drop_last=False, pin_memory=(device == 'cuda'))
ram_mb = sum(t.element_size() * t.numel() for t in (pose_w, rtg_w, act_w)) / 1e6
print(f'windows: train={n_train:,}  val={n_val:,}  (dataset RAM ≈ {ram_mb:.0f} MB)')


rows=1,053,677  sessions=549
windows: train=948,310  val=105,367  (dataset RAM ≈ 236 MB)


## 3. Train

In [3]:
from tqdm.auto import tqdm

cfg = LinearDTConfigDecoupled(
    state_dim=STATE_DIM, embed_dim=EMBED_DIM, num_actions=5,
    context_size=CONTEXT_SIZE, num_heads=NUM_HEADS, num_layers=NUM_LAYERS,
    pos_encoding=POS_ENCODING,
)
model = LinearDecisionTransformerDecoupled(cfg).to(device)
n_params = sum(p.numel() for p in model.parameters())
model = torch.compile(model)

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss()

USE_BF16 = (device == 'cuda')
print(f'params={n_params:,}  pos_encoding={POS_ENCODING}  bf16={USE_BF16}  compiled=True')

def expand_batch(rtg_b, pose_b, act_b):
    """(rtg, pose_id, action_id) windows -> (rtg, state, action_oh, targets) on device."""
    rtg_b  = rtg_b.unsqueeze(-1).to(device, non_blocking=True)            # (B, C, 1)
    pose_b = pose_b.to(device, non_blocking=True).long()                  # (B, C)
    act_b  = act_b.to(device, non_blocking=True).long()                   # (B, C)
    state  = pose_table[pose_b]                                           # (B, C, STATE_DIM)
    action = nn.functional.one_hot(act_b, num_classes=5).float()          # (B, C, 5)
    pos    = state if POS_ENCODING == 'spatial' else None
    return rtg_b, state, action, act_b, pos

history = []
t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    ep0 = time.time()
    model.train()
    sum_loss = sum_correct = sum_tokens = 0
    pbar = tqdm(train_loader, desc=f'ep {epoch:3d}/{EPOCHS} train', leave=False)
    for rtg, pose_ids, action_ids in pbar:
        rtg, state, action, targets, pos = expand_batch(rtg, pose_ids, action_ids)
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=USE_BF16):
            logits = model(rtg, state, action, pos_vecs=pos)
            loss = criterion(logits.reshape(-1, cfg.num_actions), targets.reshape(-1))
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        sum_loss    += loss.item() * targets.numel()
        sum_correct += (logits.argmax(dim=-1) == targets).sum().item()
        sum_tokens  += targets.numel()
        pbar.set_postfix(loss=f'{sum_loss/sum_tokens:.4f}', acc=f'{sum_correct/sum_tokens:.3f}')
    train_loss = sum_loss / sum_tokens; train_acc = sum_correct / sum_tokens

    model.eval()
    v_loss = v_correct = v_tokens = 0
    with torch.no_grad():
        for rtg, pose_ids, action_ids in tqdm(val_loader, desc=f'ep {epoch:3d}/{EPOCHS} val', leave=False):
            rtg, state, action, targets, pos = expand_batch(rtg, pose_ids, action_ids)
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=USE_BF16):
                logits = model(rtg, state, action, pos_vecs=pos)
                loss = criterion(logits.reshape(-1, cfg.num_actions), targets.reshape(-1))
            v_loss    += loss.item() * targets.numel()
            v_correct += (logits.argmax(dim=-1) == targets).sum().item()
            v_tokens  += targets.numel()
    val_loss = v_loss / max(v_tokens, 1); val_acc = v_correct / max(v_tokens, 1)

    ep_sec = time.time() - ep0
    history.append({'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc,
                    'val_loss': val_loss, 'val_acc': val_acc, 'epoch_sec': ep_sec})
    print(f'[epoch {epoch:3d}] train_loss={train_loss:.4f} acc={train_acc:.3f} '
          f'| val_loss={val_loss:.4f} acc={val_acc:.3f}  '
          f'({ep_sec:.1f}s/epoch, {time.time()-t0:.0f}s total)')

ckpt = RUN_DIR / 'model.pt'
state_dict = (model._orig_mod if hasattr(model, '_orig_mod') else model).state_dict()
torch.save({'state_dict': state_dict, 'cfg': cfg.__dict__,
            'arch': 'linear_decoupled'}, ckpt)
(RUN_DIR / 'metrics.jsonl').write_text(''.join(json.dumps(h)+'\n' for h in history))
print(f'\nsaved checkpoint -> {ckpt}')

params=410,373  pos_encoding=learned  bf16=True  compiled=True


ep   1/10 train:   0%|          | 0/926 [00:00<?, ?it/s]

ep   1/10 val:   0%|          | 0/103 [00:00<?, ?it/s]

[epoch   1] train_loss=0.4578 acc=0.816 | val_loss=0.3497 acc=0.848  (99.1s/epoch, 99s total)


ep   2/10 train:   0%|          | 0/926 [00:00<?, ?it/s]

...

ep   9/10 val:   0%|          | 0/103 [00:00<?, ?it/s]

[epoch   9] train_loss=0.3163 acc=0.860 | val_loss=0.3164 acc=0.860  (96.7s/epoch, 873s total)


ep  10/10 train:   0%|          | 0/926 [00:00<?, ?it/s]

ep  10/10 val:   0%|          | 0/103 [00:00<?, ?it/s]

[epoch  10] train_loss=0.3162 acc=0.860 | val_loss=0.3102 acc=0.863  (97.1s/epoch, 970s total)

saved checkpoint -> ../runs/dt/nb07/model.pt


## 4. Inference movie

In [4]:
import imageio.v2 as imageio
from PIL import Image, ImageDraw
from collections import deque

RTG_TARGET   = 20.0
MAX_STEPS    = 600
BASE_TEMP    = 1.0
ROLLOUT_SEED = 0
MOVIE_PATH   = RUN_DIR / 'rollout.mp4'
ACTION_NAMES = ['Left', 'Right', 'Forward', 'EnterWell', 'Pause']

model.eval()
env = CornerMazeEnv(session_type='exposure', obs_mode='view', render_mode='rgb_array')
env.reset(seed=ROLLOUT_SEED)

buf_state  = deque([np.zeros(STATE_DIM, dtype=np.float32)] * CONTEXT_SIZE, maxlen=CONTEXT_SIZE)
buf_action = deque([np.zeros(5,         dtype=np.float32)] * CONTEXT_SIZE, maxlen=CONTEXT_SIZE)
rtg_tensor = torch.tensor([[[RTG_TARGET]] * CONTEXT_SIZE], dtype=torch.float32, device=device)

frames = []
last_pose = None; stagnation = 0
total_reward = 0.0; n_well_rewards = 0

for step in range(MAX_STEPS):
    x, y, d = int(env.agent_pos[0]), int(env.agent_pos[1]), int(env.agent_dir)
    if (x, y, d) == last_pose: stagnation += 1
    else: stagnation = 0
    last_pose = (x, y, d)
    temp = BASE_TEMP + (stagnation // 3) * 1.5

    s_vec = encoder.encode(x, y, d)
    buf_state.append(s_vec)
    s_t = torch.from_numpy(np.stack(buf_state)).unsqueeze(0).to(device)
    a_t = torch.from_numpy(np.stack(buf_action)).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(rtg_tensor, s_t, a_t)
        probs = torch.softmax(logits[0, -1, :] / temp, dim=-1)
        action = int(torch.multinomial(probs, num_samples=1).item())

    img = env.render()
    canvas = Image.fromarray(img).resize((416, 416)).convert('RGB')
    panel = Image.new('RGB', (416, 464), 'black')
    panel.paste(canvas, (0, 48))
    draw = ImageDraw.Draw(panel)
    label = (f'step={step:3d} act={ACTION_NAMES[action]} '
             f'rtg={RTG_TARGET:.1f} ret={total_reward:+.2f} wells={n_well_rewards}')
    if stagnation > 0: label += f' stuck={stagnation} T={temp:.1f}'
    draw.text((6, 6),  label,                 fill='white')
    draw.text((6, 24), f'pose=({x},{y},{d})', fill='white')
    frames.append(np.array(panel))

    _, reward, term, trunc, _ = env.step(action)
    total_reward += reward
    if reward > 0.5: n_well_rewards += 1
    oh = np.zeros(5, dtype=np.float32); oh[action] = 1.0
    buf_action.append(oh)
    if term or trunc:
        break

print(f'rollout: {len(frames)} steps, total_reward={total_reward:+.3f}, '
      f'wells={n_well_rewards}')
imageio.mimwrite(MOVIE_PATH, frames, fps=10, codec='libx264')
print(f'wrote -> {MOVIE_PATH}')

/home/software/Documents/projects/corner-maze-rl/.venv/lib/python3.12/site-packages/torch/_inductor/compile_fx.py:322: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(


rollout: 600 steps, total_reward=+4.890, wells=5
wrote -> ../runs/dt/nb07/rollout.mp4


## 5. Watch the movie

In [5]:
from IPython.display import Video
Video(str(MOVIE_PATH), embed=False, width=500)